In [ ]:
# --- Bootstrap de Colab (no hace nada fuera de Colab) --------------------------------
# Colab arranca en /content con un entorno vacío: sin esta celda, el `import environment`
# siguiente falla con ModuleNotFoundError. La genera tools/py_to_notebook.py -- editala
# ahí, no acá.
import hashlib
import os
import subprocess
import sys

# El disco de Colab se borra cuando se desconecta el entorno de ejecución (~90 min inactivo,
# 12 h como máximo). Con USE_DRIVE = True, artifacts/ (documentos, embeddings, índices,
# resultados, cachés del LLM) vive en cambio en tu Google Drive, y al volver a correr se
# saltea todo lo que ya está construido.
USE_DRIVE = True
# Guardar también en Drive la caché de modelos de HuggingFace (~3 GB). Apagado por defecto:
# volver a descargar suele ser más rápido que leer archivos grandes a través del montaje de Drive.
MODELS_ON_DRIVE = True

# Un entorno de ejecución de Colab, desde el navegador o desde la extensión de Colab de VS Code
# (cuyo kernel puede no haber importado google.colab: la propia variable de entorno del runtime
# dice dónde estamos).
if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG"):
    REPO_URL = "https://github.com/Jahdejosefina/projecto_ragTP2.git"
    REPO_BRANCH = ""
    REPO_DIR = "projecto_rag"
    DRIVE = "/content/drive/MyDrive/projecto_rag"
    quiet = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # nunca esperar un pedido de contraseña

    def github_token():
        # Un repositorio privado: un token de GitHub como secreto de Colab llamado GITHUB_TOKEN.
        # Los secretos de Colab solo se pueden leer desde el navegador; una variable de entorno
        # funciona en cualquier lado.
        token = os.environ.get("GITHUB_TOKEN")
        if not token:
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                token = None
        return token

    # Dónde ESTAMOS antes que qué hay en disco: `isdir(REPO_DIR)` es relativo, así que
    # chequearlo primero clonaría una copia anidada cada vez que se vuelve a correr esta celda
    # en la misma sesión.
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.isdir(REPO_DIR):
            cmd = ["git", "clone", "--depth", "1"] + (["--branch", REPO_BRANCH] if REPO_BRANCH else [])
            if subprocess.run(cmd + [REPO_URL, REPO_DIR], env=quiet).returncode != 0:
                token = github_token()
                if not token:
                    raise SystemExit("git clone failed. Is the repository public? For a private one, "
                                     "add a Colab secret named GITHUB_TOKEN (in browser Colab: VS Code's "
                                     "extension can't read Colab secrets).")
                authed = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
                # capture_output: un mensaje de error no tiene que imprimir el token en el notebook
                if subprocess.run(cmd + [authed, REPO_DIR], env=quiet, capture_output=True).returncode != 0:
                    raise SystemExit("git clone failed even with GITHUB_TOKEN: check that the token "
                                     "can read this repository (fine-grained: Contents: Read).")
                # El token NO se guarda en .git/config: un pull posterior lo vuelve a pasar (abajo).
                subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])
        os.chdir(REPO_DIR)
    # Traer los arreglos publicados después de tu primer clon (no hace nada si no hay).
    pulled = subprocess.run(["git", "pull", "--ff-only", "-q"], capture_output=True, text=True, env=quiet)
    token = github_token() if pulled.returncode != 0 else None
    if token:   # un repositorio privado: volver a hacer pull con el token, sin guardarlo
        branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True,
                                text=True).stdout.strip()
        authed = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
        pulled = subprocess.run(["git", "pull", "--ff-only", "-q", authed, branch], capture_output=True,
                                text=True, env=quiet)
    if pulled.returncode != 0:
        message = (pulled.stderr or pulled.stdout).strip()
        print("note: git pull skipped:", (message.replace(token, "***") if token else message)[:200])
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

    # Instalar una vez por VM: una marca con clave en el contenido de los archivos de requirements.
    reqs = [f for f in ("requirements.txt", "requirements-group.txt") if os.path.isfile(f)]
    digest = hashlib.sha256(b"".join(open(f, "rb").read() for f in reqs)).hexdigest()[:12]
    stamp = "/content/.lab_pip_" + digest
    if not os.path.exists(stamp):
        pip = [sys.executable, "-m", "pip", "install", "-q"]
        for f in reqs:
            pip += ["-r", f]
        subprocess.run(pip, check=True)  # UNA sola resolución: una versión del grupo que choca con una del núcleo falla ruidosamente
        open(stamp, "w").close()

    if MODELS_ON_DRIVE and not USE_DRIVE:
        print("note: MODELS_ON_DRIVE needs USE_DRIVE = True; the model cache stays on the VM disk.")
    if USE_DRIVE:
        # ANTES de `import environment`: un directorio artifacts/ real creado primero le ganaría
        # en silencio al link, y todo terminaría en el disco efímero.
        if not os.path.isdir("/content/drive/MyDrive"):   # VS Code: ya montado por su comando
            # Se avisa ANTES de la llamada: en la extensión de Colab de VS Code, drive.mount() puede colgarse en vez de fallar.
            print("Mounting Drive... (in VS Code, if this hangs: interrupt, run 'Colab: Mount Google Drive "
                  "to Server...' from Ctrl+Shift+P, then re-run this cell)")
            try:
                from google.colab import drive
                drive.mount("/content/drive")
            except Exception as e:  # drive.mount() desde una celda no funciona en la extensión de Colab de VS Code
                raise SystemExit("Drive is not mounted (" + type(e).__name__ + "). In VS Code: Ctrl+Shift+P > "
                                 "'Colab: Mount Google Drive to Server...', then re-run this cell.")
        os.makedirs(DRIVE + "/artifacts", exist_ok=True)
        if os.path.islink("artifacts"):
            print("artifacts ->", os.readlink("artifacts"))
        elif os.path.isdir("artifacts"):
            print("WARNING: artifacts/ is already a real directory, so it is NOT on Drive.")
            print("Move its contents into", DRIVE + "/artifacts", ", delete it, then re-run this cell.")
        else:
            os.symlink(DRIVE + "/artifacts", "artifacts")
            print("artifacts ->", DRIVE + "/artifacts")
        os.environ["LAB_USE_DRIVE"] = "1"
        if MODELS_ON_DRIVE:
            os.makedirs(DRIVE + "/hf_cache", exist_ok=True)
            os.environ.setdefault("HF_HOME", DRIVE + "/hf_cache")

    print("Colab bootstrap OK. Working directory:", os.getcwd())

Move its contents into /content/drive/MyDrive/projecto_rag/artifacts , delete it, then re-run this cell.
Colab bootstrap OK. Working directory: /content/projecto_rag


# Etapa 5

Reutiliza **BM25 de la Etapa 3**, el módulo **`grounded_stage4.py`** y **`eval/dialogues.jsonl` congelado**. No modifica ninguno de esos insumos.

**Primero:** `SPLIT='dev'`, `SMOKE_TEST=True`, `RUN_PART_B=False`. **Después:** A completa en `dev`, luego B en `dev`; `test` solamente tras fijar las decisiones. Requiere el `documents.jsonl` y el índice persistido de Etapa 3 en el mismo Drive que las otras etapas.



In [ ]:
import environment  # antes de importar los demás módulos del proyecto
import json, hashlib, time, random, re, csv, warnings, importlib.util
from pathlib import Path
from collections import defaultdict
from dataclasses import asdict

import schema, llm
from grounded_stage4 import (find_stage3_run, load_stage3_run, GroundedAnswer,
                             Stage4Config, SYSTEM_GROUNDED, fit_prompt, contains_key)

SPLIT = 'test'            # Cambiar a 'test' solo tras fijar todas las decisiones.
SMOKE_TEST = False        # Reduce guiones; NO reduce el corpus indexado.
RUN_PART_A_LLM = True   # Parte A: la consulta reescrita por qwen3
RUN_PART_B = True       # Activar después de verificar la Parte A y la etapa 4
MODEL = 'qwen3:1.7b' if SMOKE_TEST else 'qwen3:8b'
NUM_CTX, NUM_PREDICT = 8192, 700
SEED = 1337
KS = (1, 3, 5)
# La Parte B conserva un presupuesto COMPARTIDO pasajes + historial, en tokens de Qwen:
TOTAL_SHARED_BUDGET = 500
PASSAGE_FRACTIONS = (0.8, 0.5, 0.3)
RECENT_MESSAGES = 4              # 2 pares usuario/asistente

ART = environment.artifacts_dir(False)
DOC_PATH = ART / 'documents.jsonl'
EVAL_DIR = Path('eval')

# Carpeta independiente para los resultados finales de test
OUT = ART / 'results' / 'stage5_full_test' / 'stage5_rewrite_v3'
OUT.mkdir(parents=True, exist_ok=True)


OUT.mkdir(parents=True, exist_ok=True)
assert SPLIT in ('dev', 'test')
assert not (SPLIT == 'test' and SMOKE_TEST), 'Nunca evaluar test con smoke.'
assert DOC_PATH.is_file(), f'Falta {DOC_PATH}; recuperar Etapa 1 desde Drive.'
assert (EVAL_DIR / 'FROZEN.json').is_file(), 'Falta eval/FROZEN.json en el repositorio.'
print('SPLIT:', SPLIT, '| SMOKE:', SMOKE_TEST, '| resultados:', OUT)
print("OUT:", OUT)

SPLIT: test | SMOKE: False | resultados: /content/projecto_rag/artifacts/full/results/stage5_full_test/stage5_rewrite_v3
OUT: /content/projecto_rag/artifacts/full/results/stage5_full_test/stage5_rewrite_v3


In [ ]:

from pathlib import Path
import shutil
import json

# Resultados almacenados en Google Drive

from pathlib import Path
import shutil
import json

# Ubicación de los resultados en Google Drive
drive_results = Path(
    "/content/drive/MyDrive/projecto_rag/artifacts/full/results"
)

# Ubicación local utilizada por el notebook
local_results = Path(ART) / "results"
local_results.mkdir(parents=True, exist_ok=True)

corridas = []

for carpeta in drive_results.glob("stage3_*"):
    if not carpeta.is_dir():
        continue

    spec_path = carpeta / "run_spec.json"
    report_path = carpeta / "run_report.json"

    if not (spec_path.is_file() and report_path.is_file()):
        continue

    spec = json.loads(spec_path.read_text(encoding="utf-8"))
    report = json.loads(report_path.read_text(encoding="utf-8"))

    if (
        spec.get("split") == SPLIT
        and spec.get("profile") == "full"
        and report.get("status") == "ok"
        and (carpeta / "retrieval.jsonl").is_file()
    ):
        corridas.append((report.get("finished_at") or "", carpeta))

if not corridas:
    raise FileNotFoundError(
        f"No se encontraron corridas válidas de Etapa 3 para {SPLIT}"
    )

# Seleccionar la corrida válida más reciente
corridas.sort(key=lambda x: (x[0], x[1].name))
origen_s3 = corridas[-1][1]

# Variable exclusiva para la carpeta de resultados
S3_RUN_DIR = local_results / origen_s3.name

shutil.copytree(origen_s3, S3_RUN_DIR, dirs_exist_ok=True)

print("✅ Corrida seleccionada:", origen_s3.name)
print("Split:", SPLIT)
print("Resultados:", S3_RUN_DIR)
print("retrieval.jsonl:", (S3_RUN_DIR / "retrieval.jsonl").is_file())


# Carpeta local que utiliza el notebook
local_results = Path(ART) / "results"
local_results.mkdir(parents=True, exist_ok=True)

# Buscar corridas válidas de Etapa 3 para el split actual
corridas = []

for carpeta in drive_results.glob("stage3_*"):
    if not carpeta.is_dir():
        continue

    spec_path = carpeta / "run_spec.json"
    report_path = carpeta / "run_report.json"

    if not spec_path.is_file() or not report_path.is_file():
        continue

    spec = json.loads(spec_path.read_text(encoding="utf-8"))
    report = json.loads(report_path.read_text(encoding="utf-8"))

    if (
        spec.get("split") == SPLIT
        and spec.get("profile") == "full"
        and report.get("status") == "ok"
    ):
        corridas.append((
            report.get("finished_at", ""),
            carpeta
        ))

if not corridas:
    raise FileNotFoundError(
        f"No hay corridas válidas de Etapa 3 para {SPLIT} en Drive"
    )

# Elegir la corrida con la fecha de finalización más reciente
corridas.sort(key=lambda x: (x[0], x[1].name))
origen = corridas[-1][1]
destino = local_results / origen.name

shutil.copytree(origen, destino, dirs_exist_ok=True)

print("Corrida de Etapa 3 copiada")
print("Split:", SPLIT)
print("Carpeta:", origen.name)
print("Destino:", destino)


✅ Corrida seleccionada: stage3_67ecda17a40b8c18
Split: test
Resultados: /content/projecto_rag/artifacts/full/results/stage3_67ecda17a40b8c18
retrieval.jsonl: True
Corrida de Etapa 3 copiada
Split: test
Carpeta: stage3_67ecda17a40b8c18
Destino: /content/projecto_rag/artifacts/full/results/stage3_67ecda17a40b8c18


In [ ]:
from pathlib import Path
import shutil

# Índice original en Google Drive
origen = Path(
    "/content/drive/MyDrive/projecto_rag/artifacts/full/index/stage3_36a95c9c1551289f"
)

# Ubicación que espera el notebook de Etapa 5
destino = Path(
    "/content/projecto_rag/artifacts/full/index/stage3_36a95c9c1551289f"
)

if not origen.is_dir():
    raise FileNotFoundError(f"No se encontró el índice en Drive: {origen}")

shutil.copytree(origen, destino, dirs_exist_ok=True)

print("Índice copiado correctamente")
print("Manifest:", (destino / "manifest.json").is_file())
print("Chunks:", (destino / "chunks.jsonl").is_file())
print("BM25:", (destino / "bm25").is_dir())

Índice copiado correctamente
Manifest: True
Chunks: True
BM25: True


## 1. Validar los insumos y reutilizar el índice BM25

No regeneramos el corpus ni los chunks. `load_stage3_run` verifica el hash de los chunks y la correspondencia con el índice. **La Etapa 3 tiene que haberse ejecutado con `SMOKE_TEST=False`**, al menos para `dev`.

In [ ]:
from pathlib import Path
import json

ruta = Path("/content/projecto_rag/notebooks/03_retrieval.ipynb")

nb = json.loads(ruta.read_text(encoding="utf-8"))

print("Archivo:", ruta)
print("Cantidad de celdas:", len(nb["cells"]))

for i, celda in enumerate(nb["cells"]):
    codigo = "".join(celda.get("source", []))
    if "BM25Retriever" in codigo:
        print(
            "Celda:", i,
            "| Tipo:", celda["cell_type"],
            "| Tiene declaración exacta:",
            "class BM25Retriever:" in codigo,
            "| Tiene RetrievalConfig:",
            "class RetrievalConfig" in codigo
        )

Archivo: /content/projecto_rag/notebooks/03_retrieval.ipynb
Cantidad de celdas: 39
Celda: 7 | Tipo: code | Tiene declaración exacta: True | Tiene RetrievalConfig: True
Celda: 14 | Tipo: code | Tiene declaración exacta: False | Tiene RetrievalConfig: False
Celda: 37 | Tipo: code | Tiene declaración exacta: True | Tiene RetrievalConfig: True


In [ ]:
from grounded_stage4 import read_jsonl as read_jsonl4

schema.validate_documents(DOC_PATH).show().raise_if_failed()
schema.load_eval(EVAL_DIR, documents_path=DOC_PATH)
schema.validate_eval(EVAL_DIR, DOC_PATH).show().raise_if_failed()
FROZEN = json.loads((EVAL_DIR / 'FROZEN.json').read_text(encoding='utf-8'))

S3_DIR = find_stage3_run(
    ART,
    SPLIT,
    'full',
    S3_RUN_DIR
)

S3 = load_stage3_run(S3_DIR, ART)
assert S3['spec']['corpus_hash'] == FROZEN['documents_sha256'], 'Corpus e índice incompatibles'
assert S3['manifest']['documents'] == 170, 'Esta Etapa 5 exige el índice full de Etapa 3'

all_dialogues = read_jsonl4(EVAL_DIR / 'dialogues.jsonl')
dialogues = [d for d in all_dialogues if d['split'] == SPLIT]
assert dialogues, f'No hay guiones para {SPLIT}'
if SMOKE_TEST:
    dialogues = dialogues[:1]
turns = [(d, t) for d in dialogues for t in d['turns']]


# Cargar la implementación BM25 desde el notebook de Etapa 3
# que ya está disponible en el repositorio clonado de GitHub.

import ast
import json
from pathlib import Path

stage3_path = Path("/content/projecto_rag/notebooks/03_retrieval.ipynb")

if not stage3_path.is_file():
    raise FileNotFoundError(
        f"No se encontró el notebook de Etapa 3: {stage3_path}"
    )

stage3_nb = json.loads(stage3_path.read_text(encoding="utf-8"))

# Buscar solamente las celdas que DECLARAN la clase BM25Retriever,
# no las que incluyen su código dentro de un string.
implementation_cells = []

for i, cell in enumerate(stage3_nb["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    class_names = {
        node.name
        for node in tree.body
        if isinstance(node, ast.ClassDef)
    }

    if {"BM25Retriever", "RetrievalConfig"}.issubset(class_names):
        implementation_cells.append((i, source))

if len(implementation_cells) != 1:
    raise RuntimeError(
        f"Se esperaba una única celda de implementación BM25; "
        f"se encontraron {len(implementation_cells)}: "
        f"{[i for i, _ in implementation_cells]}"
    )

cell_index, source3 = implementation_cells[0]

import sys
import types

# Crear un módulo real para que @dataclass funcione correctamente
module_name = "stage3_bm25_embedded"

module3 = types.ModuleType(module_name)
sys.modules[module_name] = module3

# Ejecutar las definiciones de la Etapa 3 dentro del módulo
exec(
    compile(source3, module_name, "exec"),
    module3.__dict__
)

mod3 = module3.__dict__

cfg3 = mod3["RetrievalConfig"](**S3["spec"]["config"])
retriever = mod3["BM25Retriever"](
    list(S3["chunks"].values()),
    cfg3
)

print("✅ BM25 cargado correctamente")
print("Celda utilizada:", cell_index)
print("Chunks:", len(S3["chunks"]))
print("Corrida de Etapa 3:", S3_DIR)


for d, t in turns:
    if t['answerable']:
        for g in t['gold']:
            assert any(c['doc_id']==g['doc_id'] and c['section_id']==g['section_id']
                       for c in S3['chunks'].values()), (d['dialogue_id'], t['turn'], g)
print('Guiones:', len(dialogues), '| turnos:', len(turns), '| chunks:', len(S3['chunks']))
print('Índice Etapa 3:', S3_DIR)

documents (documents.jsonl): OK
   documents: 170
   sections: 7070
   characters: 10685267
   with version: 0
   with date: 170
eval set (eval): OK
   single-turn items: 30
   by split: {'dev': 7, 'test': 23}
   by category: {'topical': 15, 'metadata': 7, 'identifier': 5, 'out_of_corpus': 3}
   dialogues: 8
   dialogue turns: 32
   scenarios: 3
   probes by ability: {'extraction': 3, 'multi_session': 6, 'temporal': 3, 'knowledge_update': 9, 'preference': 3, 'abstention': 3}
   yes/no rows (expected_verdict): 12


DEBUG:bm25s:Building index from IDs objects


✅ BM25 cargado correctamente
Celda utilizada: 7
Chunks: 13630
Corrida de Etapa 3: /content/projecto_rag/artifacts/full/results/stage3_67ecda17a40b8c18
Guiones: 6 | turnos: 24 | chunks: 13630
Índice Etapa 3: /content/projecto_rag/artifacts/full/results/stage3_67ecda17a40b8c18


## 2. Preparar el generador y las utilidades de medición

Las llamadas LLM usan el servidor Ollama del proyecto, `num_ctx` explícito y `think=False`. Las salidas por turno se guardan inmediatamente para poder reanudar una corrida interrumpida.

In [ ]:
from pydantic import BaseModel, Field

if RUN_PART_A_LLM or RUN_PART_B:
    INFO = llm.setup(MODEL, num_ctx=NUM_CTX)
    print('Ollama/modelo:', MODEL, '| num_ctx:', NUM_CTX)

class QueryRewrite(BaseModel):
    query: str = Field(max_length=300)
class HistorySummary(BaseModel):
    summary: str = Field(max_length=800)

def count(text):
    return llm.count_tokens(text, MODEL)
def count_messages(messages):
    return llm.count_prompt_tokens(messages, MODEL)
def model_structured(messages, out_schema, num_predict=220):
    try:
        obj, result = llm.structured(messages, out_schema, model=MODEL, num_ctx=NUM_CTX,
                                    num_predict=num_predict, temperature=0, seed=0, think=False)
        if result.error:
            raise RuntimeError(str(result.error))
        return obj, result
    except llm.ContextOverflow as exc:
        raise RuntimeError(f'CONTEXT_OVERFLOW: {exc}') from exc

def load_rows(path):
    if not path.exists(): return []
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
def save_row(path,row):
    with path.open('a',encoding='utf-8') as f:
        f.write(json.dumps(row,ensure_ascii=False)+'\n')
def unique_rows(path, keys):
    rows=load_rows(path); found=set(); out=[]
    for row in rows:
        key=tuple(row[k] for k in keys)
        assert key not in found, ('Fila repetida',key)
        found.add(key);out.append(row)
    return out

def history_before(d, turn_index):
    """Historial fijo: respuestas GUIONADAS, no respuestas generadas por cada brazo."""
    hist=[]
    for prev in d['turns'][:turn_index]:
        hist += [{'role':'user','content':prev['user']},
                 {'role':'assistant','content':prev['assistant']}]
    return hist

def retrieval_metrics(t,ranked,k=5):
    if not t['answerable']:
        return {'doc_hit':None,'section_hit':None}
    gold={(g['doc_id'],g['section_id']) for g in t['gold']}
    gdocs={x[0] for x in gold}
    top=ranked[:k]
    return {'doc_hit':int(any(c['doc_id'] in gdocs for c in top)),
            'section_hit':int(any((c['doc_id'],c['section_id']) in gold for c in top))}

def summarize(rows,metric,group='arm'):
    out=[]
    for key in sorted({r[group] for r in rows}):
        rr=[r for r in rows if r[group]==key and r.get(metric) is not None and not r.get('error')]
        out.append({'group':key,'metric':metric,'n':len(rr),
                    'mean':round(sum(float(r[metric]) for r in rr)/len(rr),4) if rr else None})
    return out

def save_csv(path,rows):
    if not rows:return
    keys=list(dict.fromkeys(k for r in rows for k in r))
    with path.open('w',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=keys);w.writeheader();w.writerows(rows)

   model        qwen3:8b
   digest       500a1f067a9f
   server       0.34.4
   num_ctx      8192
   thinking     {'honors_think_false': True, 'leaks_think_tag': False, 'thinking_only': False, 'supports_thinking': True, 'answer': 'Paris'}
   key_order    {'order': 'declaration', 'reply': '{"zeta": "alpha", "alfa": "omega"}', 'error': None}
   calibration  {'template_minus_server': [-4, -4, -4], 'offset': -4, 'repeat_prompt_tokens': (295, 295), 'repeat_cached_tokens': (269, 294), 'prompt_eval_count_includes_cached': True}
   placement    qwen3:8b: 100% GPU · 6.3 GB · context 8192
Ollama/modelo: qwen3:8b | num_ctx: 8192


## 3. Parte A · Con qué buscar

Cuatro variantes sobre los mismos turnos: último turno (`last`), todo el historial (`full_history`), reescritura LLM (`llm_rewrite`) y reescritura humana (`oracle`). Las reescrituras **no deben contestar la pregunta ni inyectar la respuesta**. Reportamos `hit@k` por tipo de dependencia y número de turno. La reescritura LLM queda disponible para auditoría humana.

In [ ]:
A_PATH = OUT/'part_a_rows.jsonl'
existing_A = {(r['dialogue_id'],r['turn'],r['arm']) for r in load_rows(A_PATH)}
for d in dialogues:
    for j,t in enumerate(d['turns']):
        hist=history_before(d,j)
        formats={
            'last':t['user'],
            'full_history':'\n'.join(f"{m['role']}: {m['content']}" for m in hist+[{'role':'user','content':t['user']}]),
            'oracle':t['oracle_rewrite'],
        }
        rewrite_error=None
        if RUN_PART_A_LLM and (d['dialogue_id'],t['turn'],'llm_rewrite') not in existing_A:
            prompt = [
                {
                    'role': 'system',
                    'content': (
                        'Sos un reescritor de consultas para un buscador de legislación argentina. '
                        'Tu única tarea es transformar la pregunta actual en una consulta '
                        'completamente autónoma, comprensible sin leer el historial. '

                        'INSTRUCCIONES: '
                        '1. Identificá el tema, producto, organismo o norma al que se refiere '
                        'la pregunta actual usando exclusivamente el historial. '
                        '2. Reemplazá pronombres, elipsis y referencias conversacionales '
                        'como "eso", "lo", "esa ley", "el del comienzo", "me lo reparan" '
                        'o "volviendo a" por el referente concreto, cuando pueda identificarse. '
                        '3. La consulta final debe poder buscarse de forma independiente: '
                        'no debe contener referencias a turnos anteriores ni expresiones '
                        'como "antes", "al principio" o "en la conversación". '
                        '4. Conservá los nombres, números de leyes y términos jurídicos '
                        'relevantes que aparezcan en el historial, pero no incorpores '
                        'información de temas anteriores que no corresponda a la pregunta. '
                        '5. Conservá exactamente la intención de la pregunta: '
                        'no cambies lo que el usuario quiere averiguar. '
                        '6. Si una referencia no puede resolverse con la información '
                        'disponible, mantené la ambigüedad en lugar de inventar detalles. '
                        '7. No respondas la pregunta, no agregues la respuesta, '
                        'conclusiones ni explicaciones. '

                        'Devolvé exclusivamente el campo query con la consulta reescrita.'
                    )
                },
                {
                    'role': 'user',
                    'content': (
                        'HISTORIAL:\n'
                        + formats['full_history']
                        + '\n\nPregunta que hay que reescribir: '
                        + t['user']
                    )
                }
            ]

            try:
                rewrite,_ = model_structured(prompt, QueryRewrite, num_predict=150)
                formats['llm_rewrite']=rewrite.query
            except Exception as exc:
                rewrite_error=str(exc)
                formats['llm_rewrite']=None
        for arm,q in formats.items():
            rowid=(d['dialogue_id'],t['turn'],arm)
            if rowid in existing_A:continue
            if q is None:
                save_row(A_PATH,{'dialogue_id':d['dialogue_id'],'turn':t['turn'],
                                 'dependency':t['depends_on'],'arm':arm,'query':None,
                                 'doc_hit@1':None,'section_hit@1':None,
                                 'doc_hit@3':None,'section_hit@3':None,
                                 'doc_hit@5':None,'section_hit@5':None,
                                 'error':rewrite_error});continue
            start=time.perf_counter()
            ranked=retriever.retrieve(q)
            elapsed=time.perf_counter()-start
            row={'dialogue_id':d['dialogue_id'],'turn':t['turn'],
                 'dependency':t['depends_on'],'arm':arm,'query':q,
                 'answerable':t['answerable'],'retrieval_s':round(elapsed,4),
                 'error':None,
                 'top_chunks':[{'chunk_id':c['chunk_id'],'doc_id':c['doc_id'],
                                'section_id':c['section_id']} for c in ranked[:5]]}
            for k in KS:
                r=retrieval_metrics(t,ranked,k)
                row[f'doc_hit@{k}']=r['doc_hit']
                row[f'section_hit@{k}']=r['section_hit']
            save_row(A_PATH,row)
print('Parte A, filas:',len(load_rows(A_PATH)))

Parte A, filas: 96


In [ ]:
A=unique_rows(A_PATH,['dialogue_id','turn','arm'])
summary_a=[]
for arm in sorted({r['arm'] for r in A}):
    for dep in ['all']+sorted({r['dependency'] for r in A}):
        for k in KS:
            relevant=[r for r in A if r['arm']==arm and (dep=='all' or r['dependency']==dep)
                      and r.get(f'section_hit@{k}') is not None and not r.get('error')]
            summary_a.append({'arm':arm,'dependency':dep,'k':k,'n':len(relevant),
                              'doc_hit':round(sum(r[f'doc_hit@{k}'] for r in relevant)/len(relevant),3) if relevant else None,
                              'section_hit':round(sum(r[f'section_hit@{k}'] for r in relevant)/len(relevant),3) if relevant else None})
save_csv(OUT/'part_a_summary.csv',summary_a)
for row in summary_a:
    if row['dependency']=='all' and row['k']==5:print(row)
print('Auditar manualmente 20 reescrituras LLM (o todas si hay menos):')
for row in [r for r in A if r['arm']=='llm_rewrite'][:20]:
    print(row['dialogue_id'], row['turn'],':',row.get('query'), 'ERROR:',row.get('error'))

{'arm': 'full_history', 'dependency': 'all', 'k': 5, 'n': 23, 'doc_hit': 0.913, 'section_hit': 0.739}
{'arm': 'last', 'dependency': 'all', 'k': 5, 'n': 23, 'doc_hit': 0.739, 'section_hit': 0.739}
{'arm': 'llm_rewrite', 'dependency': 'all', 'k': 5, 'n': 23, 'doc_hit': 0.957, 'section_hit': 0.957}
{'arm': 'oracle', 'dependency': 'all', 'k': 5, 'n': 23, 'doc_hit': 0.957, 'section_hit': 0.957}
Auditar manualmente 20 reescrituras LLM (o todas si hay menos):
d002 1 : Según la Ley 26.043, ¿qué advertencia sobre la salud debe aparecer en los envases de videojuegos? ERROR: None
d002 2 : ¿Cuál es la secretaría que aplica la Ley 26.043? ERROR: None
d002 3 : ¿Cuál es la fecha de sanción de la Ley 26.043? ERROR: None
d002 4 : ¿Cuál es la fecha de promulgación de hecho de la Ley 26.043? ERROR: None
d003 1 : ¿A partir de qué edad se aplica el Sistema Único Normalizado de Talles de Indumentaria? ERROR: None
d003 2 : ¿Cada cuánto deben repetir el estudio corporal para actualizar el Sistema Único Normal

In [ ]:

# Hit@5 por número de turno
for arm in ["last", "full_history", "llm_rewrite", "oracle"]:
    print(f"\nEstrategia: {arm}")

    for turno in sorted({r["turn"] for r in A}):
        filas = [
            r for r in A
            if r["arm"] == arm
            and r["turn"] == turno
            and r.get("section_hit@5") is not None
            and not r.get("error")
        ]

        if filas:
            aciertos = sum(r["section_hit@5"] for r in filas)
            print(
                f"Turno {turno}: "
                f"{aciertos}/{len(filas)} "
                f"({aciertos / len(filas):.1%})"
            )



Estrategia: last
Turno 1: 6/6 (100.0%)
Turno 2: 3/6 (50.0%)
Turno 3: 4/6 (66.7%)
Turno 4: 4/5 (80.0%)

Estrategia: full_history
Turno 1: 6/6 (100.0%)
Turno 2: 3/6 (50.0%)
Turno 3: 3/6 (50.0%)
Turno 4: 5/5 (100.0%)

Estrategia: llm_rewrite
Turno 1: 6/6 (100.0%)
Turno 2: 5/6 (83.3%)
Turno 3: 6/6 (100.0%)
Turno 4: 5/5 (100.0%)

Estrategia: oracle
Turno 1: 6/6 (100.0%)
Turno 2: 5/6 (83.3%)
Turno 3: 6/6 (100.0%)
Turno 4: 5/5 (100.0%)


In [44]:
# Mostrar 12 reescrituras LLM de test para auditoría manual
filas = [
    r for r in A
    if r.get("arm") == "llm_rewrite"
]

print("Reescrituras disponibles:", len(filas))

for r in filas[:12]:
    print("\n", "=" * 60)
    print("Diálogo:", r.get("dialogue_id"))
    print("Turno:", r.get("turn"))
    print("Consulta reescrita:", r.get("query"))

Reescrituras disponibles: 24

Diálogo: d002
Turno: 1
Consulta reescrita: Según la Ley 26.043, ¿qué advertencia sobre la salud debe aparecer en los envases de videojuegos?

Diálogo: d002
Turno: 2
Consulta reescrita: ¿Cuál es la secretaría que aplica la Ley 26.043?

Diálogo: d002
Turno: 3
Consulta reescrita: ¿Cuál es la fecha de sanción de la Ley 26.043?

Diálogo: d002
Turno: 4
Consulta reescrita: ¿Cuál es la fecha de promulgación de hecho de la Ley 26.043?

Diálogo: d003
Turno: 1
Consulta reescrita: ¿A partir de qué edad se aplica el Sistema Único Normalizado de Talles de Indumentaria?

Diálogo: d003
Turno: 2
Consulta reescrita: ¿Cada cuánto deben repetir el estudio corporal para actualizar el Sistema Único Normalizado de Talles de Indumentaria?

Diálogo: d003
Turno: 3
Consulta reescrita: ¿Qué advertencia sanitaria deben llevar los envases de videojuegos?

Diálogo: d003
Turno: 4
Consulta reescrita: ¿Cada cuántos años se repetía el estudio corporal para actualizar el Sistema Único Normal

## 4. Parte B · Qué historial conservar

Con `RUN_PART_B=True` comparamos **buffer completo**, **últimos dos pares** y **resumen acumulativo + turnos recientes**. En cada una, el mismo presupuesto compartido de **500 tokens** se asigna 80/20, 50/50 o 30/70 a pasajes/historial. El presupuesto del resto del prompt (sistema, pregunta, plantilla, reserva de salida) se verifica aparte.

El resumen es generado a partir de **turnos ya ocurridos y respuestas guionadas**. Solo se actualiza cuando los mensajes recientes superan su presupuesto. Si el historial completo no entra, se registra como **no factible**, no se recorta silenciosamente. Una corrida completa en `dev` puede involucrar varias decenas de llamadas; en `test` bastante más.

In [ ]:
def transcript_token_cost(msgs):
    return sum(count(f"{m['role']}: {m['content']}") for m in msgs)

def trim_recent(history,budget,max_messages=None):
    msgs=history[-max_messages:] if max_messages else list(history)
    # Mantener pares coherentes usuario/asistente.
    if msgs and msgs[0]['role']=='assistant': msgs=msgs[1:]
    while msgs and transcript_token_cost(msgs)>budget:
        msgs=msgs[2:]
    return msgs

def get_summary(history,history_budget):
    """Compresión incremental de pares guionados. Una actualización por desborde."""
    assert history_budget>0
    summary=''
    recent=[]
    updates=[]
    for pair_start in range(0,len(history),2):
        recent += history[pair_start:pair_start+2]
        provisional=([{'role':'system','content':'Resumen de conversación: '+summary}]
                     if summary else [])+recent
        if transcript_token_cost(provisional)<=history_budget:
            continue
        # Reducimos a un resumen para liberar memoria; no incluye respuestas reales del brazo.
        messages=[
            {'role':'system','content':('Resumí fielmente esta conversación en español. '
             'Conservá normas, fechas, números, referencias y los hechos que cambiaron. '
             'No inventes datos. Devolvé solo el campo summary.')},
            {'role':'user','content':('Resumen anterior:\n'+summary+'\n\nTurnos nuevos:\n'+
              '\n'.join(f"{m['role']}: {m['content']}" for m in recent))},
        ]

        # Reservamos tokens para el encabezado que acompaña al resumen
        encabezado = "system: Resumen de la conversación anterior: "
        max_resumen = max(
            0,
            history_budget - count(encabezado)
        )

        obj, _ = model_structured(
            messages,
            HistorySummary,
            num_predict=220
        )

        nuevo_resumen = obj.summary

        # Si el resumen no entra, intentamos condensarlo una vez más
        if count(nuevo_resumen) > max_resumen:
            if max_resumen < 10:
                raise RuntimeError(
                    "No es factible generar un resumen en este presupuesto."
                )

            prompt_corto = [
                {
                    'role': 'system',
                    'content': (
                        'Reducí este resumen manteniendo únicamente '
                        'los hechos esenciales para continuar la conversación. '
                        'No inventes información. '
                        f'Usá como máximo {max_resumen} tokens.'
                    )
                },
                {
                    'role': 'user',
                    'content': nuevo_resumen
                }
            ]

            obj_corto, _ = model_structured(
                prompt_corto,
                HistorySummary,
                num_predict=min(220, max_resumen + 30)
            )

            nuevo_resumen = obj_corto.summary

        summary = nuevo_resumen

        # Validación efectiva, incluyendo encabezado
        mensaje_resumen = [
            {
                'role': 'system',
                'content': 'Resumen de la conversación anterior: ' + summary
            }
        ]

        if transcript_token_cost(mensaje_resumen) > history_budget:
            raise RuntimeError(
                "No es factible comprimir el resumen al presupuesto."
            )

        updates.append({
            'pair': pair_start // 2 + 1,
            'summary': summary
        })

        recent = []

    # Si la síntesis sola supera el presupuesto, comunicarlo y no recortar en silencio.
    summ_msg=[{'role':'system','content':'Resumen de la conversación anterior: '+summary}] if summary else []
    if transcript_token_cost(summ_msg)>history_budget:
        raise RuntimeError('El resumen excede el presupuesto asignado.')
    recent=trim_recent(recent, max(0,history_budget-transcript_token_cost(summ_msg)))
    return summ_msg+recent, updates, summary

def history_strategy(policy,history,budget):
    if policy=='full':
        if transcript_token_cost(history)>budget:
            return None,[],''
        return list(history),[],''
    if policy=='recent':return trim_recent(history,budget,RECENT_MESSAGES),[],''
    if policy=='summary':return get_summary(history,budget)
    raise ValueError(policy)

# Ver de antemano si el historial realmente ejerce presión sobre los distintos presupuestos.
for frac in PASSAGE_FRACTIONS:
    hb=round(TOTAL_SHARED_BUDGET*(1-frac))
    lens=[transcript_token_cost(history_before(d,i)) for d in dialogues for i in range(len(d['turns']))]
    print(f'Pasajes {frac:.0%} / historial {1-frac:.0%}: historial {hb} tokens, '
          f'turnos con historia que excede = {sum(x>hb for x in lens)}/{len(lens)}, '
          f'máximo observado = {max(lens)}')

Pasajes 80% / historial 20%: historial 100 tokens, turnos con historia que excede = 7/24, máximo observado = 147
Pasajes 50% / historial 50%: historial 250 tokens, turnos con historia que excede = 0/24, máximo observado = 147
Pasajes 30% / historial 70%: historial 350 tokens, turnos con historia que excede = 0/24, máximo observado = 147


In [ ]:
B_PATH = OUT/'part_b_rows.jsonl'

# Consultas elegidas a partir de la Parte A
consultas_llm = {
    (r['dialogue_id'], r['turn']): r['query']
    for r in A
    if r['arm'] == 'llm_rewrite'
    and not r.get('error')
    and r.get('query')
}

if not RUN_PART_B:
    print('Parte B desactivada. Cuando la Parte A esté revisada, fijá RUN_PART_B=True y ejecutá esta celda.')
else:
    existing_B={(r['dialogue_id'],r['turn'],r['policy'],r['passage_fraction']) for r in load_rows(B_PATH)}
    cfg4=Stage4Config(model=MODEL,num_ctx=NUM_CTX,num_predict=NUM_PREDICT,
                      passage_budget=TOTAL_SHARED_BUDGET)
    for d in dialogues:
        for j,t in enumerate(d['turns']):
            script_hist=history_before(d,j)
            # Misma consulta elegida antes del test: 'oracle' es solo una cota,
            # no elegirla como estrategia desplegable.

            key_query = (d['dialogue_id'], t['turn'])

            assert key_query in consultas_llm, (
                'Falta la reescritura LLM', key_query
            )

            query = consultas_llm[key_query]
            ranked = retriever.retrieve(query)

            for frac in PASSAGE_FRACTIONS:
                p_budget=round(TOTAL_SHARED_BUDGET*frac)
                h_budget=TOTAL_SHARED_BUDGET-p_budget
                for policy in ('full','recent','summary'):
                    key=(d['dialogue_id'],t['turn'],policy,frac)
                    if key in existing_B:continue
                    common={'dialogue_id':d['dialogue_id'],'turn':t['turn'],
                            'dependency':t['depends_on'],'policy':policy,
                            'passage_fraction':frac,'history_budget':h_budget,
                            'passage_budget':p_budget,'answerable':t['answerable']}
                    try:
                        hist,updates,summary=history_strategy(policy,script_hist,h_budget)
                        if hist is None:
                            save_row(B_PATH,{**common,'status':'not_feasible','error':None,
                                             'correct':None,'updates':0});continue
                        hp=transcript_token_cost(hist)
                        assert hp<=h_budget, ('historial fuera de presupuesto',hp,h_budget)
                        fitted=fit_prompt(SYSTEM_GROUNDED, query, ranked, cfg4, count,
                                          count_messages, history=hist,passage_budget=p_budget)
                        if not fitted.fits:
                            raise RuntimeError('El prompt completo supera num_ctx-reserva; no se trunca.')
                        t0=time.perf_counter()
                        answer,r=model_structured(fitted.messages,GroundedAnswer,
                                                   num_predict=NUM_PREDICT)
                        elapsed=time.perf_counter()-t0
                        if not t['answerable']:
                            correct=not answer.can_answer
                        else:
                            verdict=(t.get('expected_verdict') or '').lower()
                            verdict_ok=(not verdict or verdict in ('none','no_aplica') or
                                        answer.verdict==({'yes':'si','no':'no'}.get(verdict,verdict)))
                            correct=bool(answer.can_answer and contains_key(answer.answer,t.get('answer_key')) and verdict_ok)
                        save_row(B_PATH,{**common,'status':'ok','error':None,
                                         'correct':int(correct),'can_answer':answer.can_answer,
                                         'answer':answer.answer,'citations':answer.citations,
                                         'history_tokens':hp,'passage_tokens':fitted.passage_tokens,
                                         'prompt_tokens_est':fitted.expected_prompt_tokens,
                                         'prompt_tokens_server':r.prompt_tokens,
                                         'latency_s':round(elapsed,3),'llm_calls':1+len(updates),
                                         'updates':len(updates),'summary':summary,
                                         'summary_updates':updates,
                                         'delivered_sections':[p['section_id'] for p in fitted.delivered]})
                        print(d['dialogue_id'],t['turn'],policy,frac,'correct=',correct)

                    except Exception as exc:
                        mensaje = str(exc)

                        estado = (
                            'not_feasible'
                            if 'No es factible' in mensaje
                            else 'error'
                        )

                        save_row(
                            B_PATH,
                            {
                                **common,
                                'status': estado,
                                'error': mensaje,
                                'correct': None,
                                'updates': None
                            }
                        )

                        print(estado.upper(), key, mensaje)


In [ ]:
B=load_rows(B_PATH)
if B:
    summary_b=[]
    for policy in ('full','recent','summary'):
        for frac in PASSAGE_FRACTIONS:
            sub=[r for r in B if r['policy']==policy and r['passage_fraction']==frac]
            ok=[r for r in sub if r['status']=='ok']
            summary_b.append({'policy':policy,'passage_fraction':frac,
                              'n_total':len(sub),'n_ok':len(ok),
                              'n_infeasible':sum(r['status']=='not_feasible' for r in sub),
                              'n_error':sum(r['status']=='error' for r in sub),
                              'accuracy':round(sum(r['correct'] for r in ok)/len(ok),3) if ok else None,
                              'mean_prompt_tokens':round(sum(r['prompt_tokens_est'] for r in ok)/len(ok),1) if ok else None,
                              'mean_llm_calls':round(sum(r['llm_calls'] for r in ok)/len(ok),2) if ok else None})
    save_csv(OUT/'part_b_summary.csv',summary_b)
    for row in summary_b:print(row)
    summary_snapshots=[{'dialogue_id':r['dialogue_id'],'turn':r['turn'],
                        'passage_fraction':r['passage_fraction'],'summary':r.get('summary'),
                        'updates':r.get('summary_updates')}
                       for r in B if r['policy']=='summary' and r['status']=='ok']
    (OUT/'summary_audit.json').write_text(json.dumps(summary_snapshots,ensure_ascii=False,indent=2),encoding='utf-8')

{'policy': 'full', 'passage_fraction': 0.8, 'n_total': 24, 'n_ok': 17, 'n_infeasible': 7, 'n_error': 0, 'accuracy': 0.706, 'mean_prompt_tokens': 777.2, 'mean_llm_calls': 1.0}
{'policy': 'full', 'passage_fraction': 0.5, 'n_total': 24, 'n_ok': 24, 'n_infeasible': 0, 'n_error': 0, 'accuracy': 0.667, 'mean_prompt_tokens': 655.5, 'mean_llm_calls': 1.0}
{'policy': 'full', 'passage_fraction': 0.3, 'n_total': 24, 'n_ok': 24, 'n_infeasible': 0, 'n_error': 0, 'accuracy': 0.167, 'mean_prompt_tokens': 553.2, 'mean_llm_calls': 1.0}
{'policy': 'recent', 'passage_fraction': 0.8, 'n_total': 24, 'n_ok': 24, 'n_infeasible': 0, 'n_error': 0, 'accuracy': 0.792, 'mean_prompt_tokens': 791.5, 'mean_llm_calls': 1.0}
{'policy': 'recent', 'passage_fraction': 0.5, 'n_total': 24, 'n_ok': 24, 'n_infeasible': 0, 'n_error': 0, 'accuracy': 0.667, 'mean_prompt_tokens': 641.5, 'mean_llm_calls': 1.0}
{'policy': 'recent', 'passage_fraction': 0.3, 'n_total': 24, 'n_ok': 24, 'n_infeasible': 0, 'n_error': 0, 'accuracy': 0.1

In [ ]:
# Intervalos bootstrap por GUION (no por turno): dependencia entre turnos del mismo guion.
def bootstrap_by_dialogue(rows, metric='correct', B=1000, seed=1337):
    good=[r for r in rows if r.get(metric) is not None and r.get('status')=='ok']
    groups=defaultdict(list)
    for r in good:groups[r['dialogue_id']].append(float(r[metric]))
    ids=sorted(groups)
    if not ids:return None
    rng=random.Random(seed)
    vals=[]
    for _ in range(B):
        sampled=rng.choices(ids,k=len(ids))
        numbers=[x for gid in sampled for x in groups[gid]]
        vals.append(sum(numbers)/len(numbers))
    vals.sort()
    return {'n_dialogues':len(ids),'n_turns':len(good),
            'mean':round(sum(r[metric] for r in good)/len(good),3),
            'ci95':[round(vals[int(.025*(B-1))],3),round(vals[int(.975*(B-1))],3)]}

if B:
    confidence=[]
    for policy in ('full','recent','summary'):
        for frac in PASSAGE_FRACTIONS:
            sub=[r for r in B if r['policy']==policy and r['passage_fraction']==frac]
            confidence.append({'policy':policy,'passage_fraction':frac,
                               'bootstrap_by_dialogue':bootstrap_by_dialogue(sub)})
    (OUT/'part_b_bootstrap.json').write_text(json.dumps(confidence,ensure_ascii=False,indent=2),encoding='utf-8')
    for row in confidence:print(row)


{'policy': 'full', 'passage_fraction': 0.8, 'bootstrap_by_dialogue': {'n_dialogues': 6, 'n_turns': 17, 'mean': 0.706, 'ci95': [0.5, 0.889]}}
{'policy': 'full', 'passage_fraction': 0.5, 'bootstrap_by_dialogue': {'n_dialogues': 6, 'n_turns': 24, 'mean': 0.667, 'ci95': [0.458, 0.875]}}
{'policy': 'full', 'passage_fraction': 0.3, 'bootstrap_by_dialogue': {'n_dialogues': 6, 'n_turns': 24, 'mean': 0.167, 'ci95': [0.083, 0.25]}}
{'policy': 'recent', 'passage_fraction': 0.8, 'bootstrap_by_dialogue': {'n_dialogues': 6, 'n_turns': 24, 'mean': 0.792, 'ci95': [0.667, 0.917]}}
{'policy': 'recent', 'passage_fraction': 0.5, 'bootstrap_by_dialogue': {'n_dialogues': 6, 'n_turns': 24, 'mean': 0.667, 'ci95': [0.458, 0.875]}}
{'policy': 'recent', 'passage_fraction': 0.3, 'bootstrap_by_dialogue': {'n_dialogues': 6, 'n_turns': 24, 'mean': 0.125, 'ci95': [0.042, 0.208]}}
{'policy': 'summary', 'passage_fraction': 0.8, 'bootstrap_by_dialogue': {'n_dialogues': 6, 'n_turns': 24, 'mean': 0.792, 'ci95': [0.667, 0.


### Parte B.1 — Auditoría de los resúmenes generados

Verificar qué información conserva la estrategia de resumen acumulativo (`summary`) cuando se reduce el historial de conversación.

Se inspeccionan los resúmenes generados para la configuración 80 % de pasajes y 20 % de historial, que es la que presenta mayor presión sobre la memoria conversacional.

Esta inspección permite detectar tres tipos de problemas:
- **Hechos inventados:** información que no aparece en la conversación.
- **Relaciones incorrectas:** datos atribuidos a una norma o situación equivocada.
- **Omisiones:** información relevante que desaparece durante la compresión.




In [ ]:

# Inspección de los resúmenes realmente generados
for r in B:
    if (
        r["policy"] == "summary"
        and r["status"] == "ok"
        and r.get("summary")
        and r.get("passage_fraction") == 0.8
    ):
        print("\n" + "=" * 60)
        print("Diálogo:", r["dialogue_id"])
        print("Turno:", r["turn"])
        print("RESUMEN:")
        print(r["summary"])
        print("ACTUALIZACIONES:", r["updates"])



Diálogo: d002
Turno: 4
RESUMEN:
Según la Ley 26.043, los envases de videojuegos deben incluir la advertencia «La sobreexposición es perjudicial para la salud». La ley es aplicada por la Secretaría de Defensa de la Competencia, la Desregulación y de la Defensa del Consumidor. Fue sancionada el 1 de junio de 2005.
ACTUALIZACIONES: 1

Diálogo: d003
Turno: 4
RESUMEN:
El Sistema Único Normalizado de Talles de Indumentaria se aplica a partir de los doce años y debe repetirse cada diez años. Los envases de videojuegos deben llevar la advertencia sanitaria: «La sobreexposición es perjudicial para la salud».
ACTUALIZACIONES: 1

Diálogo: d004
Turno: 4
RESUMEN:
Según la Ley 25.649, los medicamentos deben identificarse con el nombre genérico o denominación común internacional, y esta ley fue sancionada el 28 de agosto de 2002. En cuanto al turismo estudiantil, los viajes escolares que incluyen actividades del programa educativo se denominan viajes de estudios.
ACTUALIZACIONES: 1

Diálogo: d006
Tu


### Parte B.2 — Comparación con el historial original

Comprobar manualmente si los resúmenes anteriores son fieles a la conversación.

Se recuperan los mensajes originales de los diálogos `d001` y `d005`, anteriores al cuarto turno, para contrastarlos con los resúmenes generados.

In [ ]:

# Comparar los resúmenes con el historial original
# Funciona automáticamente para dev y test.

print("Split actual:", SPLIT)

for d in dialogues:
    dialogue_id = d["dialogue_id"]

    print("\n" + "=" * 60)
    print("DIÁLOGO:", dialogue_id)
    print("HISTORIAL ANTES DEL TURNO 4:\n")

    historial = history_before(d, 3)

    for mensaje in historial:
        print(
            f"{mensaje['role'].upper()}: "
            f"{mensaje['content']}\n"
        )


Split actual: test

DIÁLOGO: d002
HISTORIAL ANTES DEL TURNO 4:

USER: Según la Ley 26.043, ¿qué advertencia sobre la salud debe aparecer en los envases de videojuegos?

ASSISTANT: La advertencia es: «La sobreexposición es perjudicial para la salud».

USER: ¿Y qué secretaría aplica esa ley?

ASSISTANT: El texto designa a la Secretaría de Defensa de la Competencia, la Desregulación y de la Defensa del Consumidor.

USER: ¿Fecha de sanción?

ASSISTANT: Fue sancionada el 1 de junio de 2005.


DIÁLOGO: d003
HISTORIAL ANTES DEL TURNO 4:

USER: ¿A partir de qué edad se aplica el Sistema Único Normalizado de Talles de Indumentaria?

ASSISTANT: La población destinataria se contempla a partir de los doce años.

USER: ¿Cada cuánto deben repetir el estudio corporal para actualizar ese sistema?

ASSISTANT: Cada diez años.

USER: Cambiando de tema: ¿qué advertencia sanitaria deben llevar los envases de videojuegos?

ASSISTANT: «La sobreexposición es perjudicial para la salud».


DIÁLOGO: d004
HISTORI

In [ ]:
from grounded_stage4 import read_jsonl as read_jsonl4

todos = read_jsonl4(EVAL_DIR / "dialogues.jsonl")

for d in todos:
    if d["split"] != "test":
        continue

    print("\n" + "=" * 60)
    print("DIÁLOGO:", d["dialogue_id"])

    for t in d["turns"][:2]:
        print("\nTurno:", t["turn"])
        print("Usuario:", t["user"])
        print("Asistente:", t["assistant"])


DIÁLOGO: d002

Turno: 1
Usuario: Según la Ley 26.043, ¿qué advertencia sobre la salud debe aparecer en los envases de videojuegos?
Asistente: La advertencia es: «La sobreexposición es perjudicial para la salud».

Turno: 2
Usuario: ¿Y qué secretaría aplica esa ley?
Asistente: El texto designa a la Secretaría de Defensa de la Competencia, la Desregulación y de la Defensa del Consumidor.

DIÁLOGO: d003

Turno: 1
Usuario: ¿A partir de qué edad se aplica el Sistema Único Normalizado de Talles de Indumentaria?
Asistente: La población destinataria se contempla a partir de los doce años.

Turno: 2
Usuario: ¿Cada cuánto deben repetir el estudio corporal para actualizar ese sistema?
Asistente: Cada diez años.

DIÁLOGO: d004

Turno: 1
Usuario: Según la Ley 25.649, ¿con qué denominación deben identificarse los medicamentos en las recetas?
Asistente: Debe expresarse el nombre genérico o denominación común internacional.

Turno: 2
Usuario: ¿Fecha de sanción de esa ley?
Asistente: Fue sancionada el 


### Parte B.3 — Prueba de recuerdo de información antigua

Evaluar si las estrategias de historial permiten recuperar información mencionada al comienzo de una conversación.

Se comparan tres políticas:
- `full`: conservar todo el historial, siempre que entre en el presupuesto.
- `recent`: conservar únicamente los mensajes recientes.
- `summary`: conservar información anterior mediante un resumen acumulativo.

Para cada diálogo se realiza una pregunta sobre un dato del primer turno, utilizando exclusivamente el historial conservado y sin proporcionar pasajes del corpus.

Esto permite aislar la capacidad de recordar información conversacional de la capacidad de recuperarla nuevamente desde los documentos.

Se registra si cada respuesta es correcta y qué configuraciones resultan inviables por sus restricciones de tokens.


In [ ]:

# Se evalúa memoria conversacional, SIN recuperar pasajes del corpus.

import unicodedata
from pydantic import BaseModel, Field

class RecallAnswer(BaseModel):
    answer: str = Field(max_length=300)

def normalizar_respuesta(texto):
    texto = unicodedata.normalize("NFD", texto.lower())
    return "".join(
        c for c in texto
        if unicodedata.category(c) != "Mn"
    )

# Preguntas sobre hechos mencionados al principio de cada diálogo.

probes = {
    "d001": {
        "question": "¿Qué duración de garantía mencionó el asistente al comienzo de esta conversación para el bien usado?",
        "expected": ["tres meses", "3 meses"]
    },
    "d005": {
        "question": "¿Qué dato debía mencionar la institución educativa en su publicidad, según el comienzo de la conversación?",
        "expected": ["numero de resolucion", "numero de la resolucion"]
    },
    "d002": {
        "question": "¿Qué advertencia sanitaria mencionó el asistente al principio de la conversación?",
        "expected": ["la sobreexposicion es perjudicial para la salud"]
    },
    "d003": {
        "question": "¿Desde qué edad dijo el asistente que se aplica el sistema de talles?",
        "expected": ["doce años", "12 años"]
    },
    "d004": {
        "question": "¿Con qué denominación indicó el asistente que se identifican los medicamentos?",
        "expected": ["nombre generico", "denominacion comun internacional"]
    },
    "d006": {
        "question": "¿Cuántos días mencionó el asistente como plazo para que la empresa demuestre que no fue responsable de la interrupción?",
        "expected": ["treinta dias", "30 dias"]
    },
    "d007": {
        "question": "¿Qué fecha de promulgación mencionó el asistente en la primera respuesta?",
        "expected": ["8 de junio de 2016", "08 de junio de 2016"]
    },
    "d008": {
        "question": "¿En qué idioma dijo el asistente que debía estar redactado el documento de venta?",
        "expected": ["castellano"]
    }
}


for d in dialogues:
    assert d["dialogue_id"] in probes, (
        f"Falta sonda para {d['dialogue_id']}"
    )
    assert len(d["turns"]) >= 3, (
        f"El diálogo {d['dialogue_id']} tiene menos de 3 turnos"
    )

RECALL_PATH = OUT / "early_recall.csv"
recall_results = []

for d in dialogues:
    did = d["dialogue_id"]

    if did not in probes:
        continue

    # Usamos los tres primeros turnos completos como historial.
    historial_original = history_before(d, 3)
    probe = probes[did]

    for frac in PASSAGE_FRACTIONS:

        h_budget = round(TOTAL_SHARED_BUDGET * (1 - frac))

        for policy in ("full", "recent", "summary"):

            fila = {
                "dialogue_id": did,
                "policy": policy,
                "passage_fraction": frac,
                "history_budget": h_budget,
                "status": None,
                "correct": None,
                "answer": None
            }

            try:
                hist, updates, resumen = history_strategy(
                    policy,
                    historial_original,
                    h_budget
                )

                if hist is None:
                    fila["status"] = "not_feasible"

                else:
                    assert transcript_token_cost(hist) <= h_budget

                    messages = [
                        {
                            "role": "system",
                            "content": (
                                "Respondé usando exclusivamente la "
                                "información del historial de conversación. "
                                "No uses conocimientos externos. "
                                "Si el dato no está disponible, respondé "
                                "'No lo recuerdo'."
                            )
                        },
                        *hist,
                        {
                            "role": "user",
                            "content": probe["question"]
                        }
                    ]

                    obj, _ = model_structured(
                        messages,
                        RecallAnswer,
                        num_predict=100
                    )

                    respuesta = normalizar_respuesta(obj.answer)

                    correcto = any(
                        normalizar_respuesta(k) in respuesta
                        for k in probe["expected"]
                    )

                    fila.update({
                        "status": "ok",
                        "correct": int(correcto),
                        "answer": obj.answer
                    })

            except Exception as exc:
                fila["status"] = "error"
                fila["answer"] = str(exc)

            recall_results.append(fila)

            print(
                did,
                "|", policy,
                "| pasajes:", frac,
                "| estado:", fila["status"],
                "| correcto:", fila["correct"],
                "| respuesta:", fila["answer"]
            )

# Guardar resultados
save_csv(RECALL_PATH, recall_results)

print("\nResultados guardados en:", RECALL_PATH)


d002 | full | pasajes: 0.8 | estado: not_feasible | correcto: None | respuesta: None
d002 | recent | pasajes: 0.8 | estado: ok | correcto: 0 | respuesta: No lo recuerdo
d002 | summary | pasajes: 0.8 | estado: ok | correcto: 1 | respuesta: La advertencia sanitaria mencionada al principio de la conversación fue: «La sobreexposición es perjudicial para la salud».
d002 | full | pasajes: 0.5 | estado: ok | correcto: 1 | respuesta: La advertencia sanitaria mencionada al principio de la conversación fue: «La sobreexposición es perjudicial para la salud».
d002 | recent | pasajes: 0.5 | estado: ok | correcto: 0 | respuesta: No lo recuerdo
d002 | summary | pasajes: 0.5 | estado: ok | correcto: 1 | respuesta: La advertencia sanitaria mencionada al principio de la conversación fue: «La sobreexposición es perjudicial para la salud».
d002 | full | pasajes: 0.3 | estado: ok | correcto: 1 | respuesta: La advertencia sanitaria mencionada al principio de la conversación fue: «La sobreexposición es perju

In [43]:

# - Resumen de resultados de recuerdo

print("RESULTADOS DE RECUERDO TEMPRANO\n")

for policy in ("full", "recent", "summary"):
    for frac in PASSAGE_FRACTIONS:

        filas = [
            r for r in recall_results
            if r["policy"] == policy
            and r["passage_fraction"] == frac
        ]

        validas = [r for r in filas if r["status"] == "ok"]
        aciertos = sum(r["correct"] for r in validas)

        accuracy = (
            round(aciertos / len(validas), 3)
            if validas else None
        )

        print(
            f"{policy:8s} | pasajes={frac:.0%} "
            f"| correctas={aciertos}/{len(validas)} "
            f"| accuracy={accuracy} "
            f"| inviables={sum(r['status'] == 'not_feasible' for r in filas)} "
            f"| errores={sum(r['status'] == 'error' for r in filas)}"
        )


RESULTADOS DE RECUERDO TEMPRANO

full     | pasajes=80% | correctas=0/0 | accuracy=None | inviables=6 | errores=0
full     | pasajes=50% | correctas=6/6 | accuracy=1.0 | inviables=0 | errores=0
full     | pasajes=30% | correctas=6/6 | accuracy=1.0 | inviables=0 | errores=0
recent   | pasajes=80% | correctas=1/6 | accuracy=0.167 | inviables=0 | errores=0
recent   | pasajes=50% | correctas=1/6 | accuracy=0.167 | inviables=0 | errores=0
recent   | pasajes=30% | correctas=1/6 | accuracy=0.167 | inviables=0 | errores=0
summary  | pasajes=80% | correctas=6/6 | accuracy=1.0 | inviables=0 | errores=0
summary  | pasajes=50% | correctas=6/6 | accuracy=1.0 | inviables=0 | errores=0
summary  | pasajes=30% | correctas=6/6 | accuracy=1.0 | inviables=0 | errores=0


### Parte B.4 — Diagnóstico de pérdida de información en la ventana reciente

In [42]:

# Diagnóstico de pérdida de información en la ventana reciente
# Funciona tanto para dev como para test.

print("Split actual:", SPLIT)
print("Límite de mensajes recientes:", RECENT_MESSAGES)

for d in dialogues:
    dialogue_id = d["dialogue_id"]

    # Historial anterior al cuarto turno
    if len(d["turns"]) < 4:
        continue

    historial = history_before(d, 3)

    print("\n" + "=" * 60)
    print("DIÁLOGO:", dialogue_id)
    print("Tokens del historial completo:",
          transcript_token_cost(historial))

    for presupuesto in (100, 250, 350):
        conservados = trim_recent(
            historial,
            presupuesto,
            RECENT_MESSAGES
        )

        print(f"\n--- Presupuesto: {presupuesto} tokens ---")
        print("Mensajes conservados:", len(conservados))
        print("Tokens utilizados:",
              transcript_token_cost(conservados))

        for m in conservados:
            print(m["role"].upper(), ":", m["content"])


Split actual: test
Límite de mensajes recientes: 4

DIÁLOGO: d002
Tokens del historial completo: 126

--- Presupuesto: 100 tokens ---
Mensajes conservados: 4
Tokens utilizados: 74
USER : ¿Y qué secretaría aplica esa ley?
ASSISTANT : El texto designa a la Secretaría de Defensa de la Competencia, la Desregulación y de la Defensa del Consumidor.
USER : ¿Fecha de sanción?
ASSISTANT : Fue sancionada el 1 de junio de 2005.

--- Presupuesto: 250 tokens ---
Mensajes conservados: 4
Tokens utilizados: 74
USER : ¿Y qué secretaría aplica esa ley?
ASSISTANT : El texto designa a la Secretaría de Defensa de la Competencia, la Desregulación y de la Defensa del Consumidor.
USER : ¿Fecha de sanción?
ASSISTANT : Fue sancionada el 1 de junio de 2005.

--- Presupuesto: 350 tokens ---
Mensajes conservados: 4
Tokens utilizados: 74
USER : ¿Y qué secretaría aplica esa ley?
ASSISTANT : El texto designa a la Secretaría de Defensa de la Competencia, la Desregulación y de la Defensa del Consumidor.
USER : ¿Fecha d

## 5. Informe y control de reproducibilidad

Guardá los CSV y JSONL de resultados. Antes de publicar conclusiones: auditá al menos 20 reescrituras, verificá que las políticas de historial hayan perdido información en una fracción significativa de los turnos, y revisá los resúmenes por **hechos inventados, relaciones equivocadas y detalles faltantes**. El bootstrap y comparaciones deben agrupar por **guion**, no tratar turnos del mismo guion como independientes.

**Nota:** esta primera versión no incorpora una sonda pareada específica para comprobar el recuerdo de hechos de los primeros turnos. No informes ese experimento como realizado hasta implementarlo y ejecutarlo.

In [ ]:
report={'stage':'05_conversation','split':SPLIT,'smoke':SMOKE_TEST,
        'model':MODEL,'num_ctx':NUM_CTX,'num_predict':NUM_PREDICT,
        'stage3_run':str(S3_DIR),'stage3_index_key':S3['spec']['index_key'],
        'corpus_hash':FROZEN['documents_sha256'],
        'dialogues':[d['dialogue_id'] for d in dialogues],
        'part_a_rows':len(load_rows(A_PATH)),'part_b_rows':len(load_rows(B_PATH)),
        'passage_fractions':PASSAGE_FRACTIONS,'shared_budget':TOTAL_SHARED_BUDGET,
        'pending': ['evaluación final en test']
}
(OUT/'run_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print('RESULTADOS:',OUT)
for p in sorted(OUT.iterdir()):print(' -',p.name)

In [ ]:
from pathlib import Path
import shutil

origen = Path(ART) / "results" / "stage5_full_test" / "stage5_rewrite_v3"

destino = Path(
    "/content/drive/MyDrive/projecto_rag/artifacts/full/results/stage5_full_test/stage5_rewrite_v3"
)

if not origen.is_dir():
    raise FileNotFoundError(origen)

shutil.copytree(origen, destino, dirs_exist_ok=True)

print(" --- Resultados de test guardados en Drive ---")
for archivo in sorted(destino.iterdir()):
    print("-", archivo.name)

 --- Resultados de test guardados en Drive ---
- early_recall.csv
- part_a_rows.jsonl
- part_a_summary.csv
- part_b_bootstrap.json
- part_b_rows.jsonl
- part_b_summary.csv
- run_report.json
- summary_audit.json
